In [6]:
import sys
sys.path.append("/kaggle/input/enedis-src/src")
from pathlib import Path
import pandas as pd
import numpy as np


from models.BiLSTM import BiLSTMImputer
from train.pipeline import train_bilstm
from config import ModelConfig



## Import des données

In [7]:
DATA_PATH = Path("../input/preprocessed")
X_tr = pd.read_csv(DATA_PATH / "X_train.csv", index_col=0,parse_dates=True)
X_test = pd.read_csv(DATA_PATH / "X_test.csv", index_col=0,parse_dates=True)
Y_tr = pd.read_csv(DATA_PATH / "y_train.csv", index_col=0,parse_dates=True)

In [8]:
X_tr = np.expm1(X_tr)
X_test = np.expm1(X_test)
Y_tr = np.expm1(Y_tr)

/usr/local/lib/python3.11/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in expm1
  result = func(self.values, **kwargs)


In [9]:
holed_cols = [c for c in X_tr.columns if c.startswith("holed")]

## Entraînement

In [ ]:
##### === Configuration ===
config = ModelConfig()

config.use_interpolation = True
config.use_rolling_stats = True      
config.use_hourly_profile = True    
config.use_temporal = False         
print(config.device)
config.learning_rate = 0.001
print(config.learning_rate)

clean_cols_train = [c for c in X_tr.columns if not c.startswith("holed")]
clean_cols_test = [c for c in X_test.columns if not c.startswith("holed")]
X_all_clean = pd.concat([
    X_tr[clean_cols_train],
    X_test[clean_cols_test]
], axis=1)

config.build_feature_extractor(X_train_clean=X_all_clean)
 
print(config.feature_extractor)

# === Créer le modèle ===
input_size = config.feature_extractor.get_input_size()

model = BiLSTMImputer(
    input_size=input_size,  
    hidden_size=config.hidden_size,
    num_layers=config.num_layers,
    dropout=config.dropout
).to(config.device)

print(config.device)

# === Entraîner ===
model, scaler = train_bilstm(X_tr, X_test, Y_tr, holed_cols, config,n_epochs_pretrain=20)

Device set to : cuda
Check
cuda
0.001

✓ Feature extractor construit :
  FeatureExtractor(10 dims: value, mask, interpolation, rolling_stats_w6_12, hourly_profile)
  [1] value (dim=1)
  [2] mask (dim=1)
  [3] interpolation (dim=1)
  [4] rolling_stats_w6_12 (dim=6)
  [5] hourly_profile (dim=1)
FeatureExtractor(10 dims: value, mask, interpolation, rolling_stats_w6_12, hourly_profile)
cuda

 ENTRAÎNEMENT BiLSTM
   Features : FeatureExtractor(10 dims: value, mask, interpolation, rolling_stats_w6_12, hourly_profile)
   Fine-tuning activé : True
 CUDA available : True
 Device check : cuda

✓ Modèle : BiLSTMImputer
  Input size : 10
  Paramètres : 934,145

✓ Model device : cuda:0

 Données :
  - Total courbes complètes : 57,140
  - Courbes à trous : 999

PHASE 1 : PRÉ-ENTRAÎNEMENT GÉNÉRIQUE
Nombre de courbes complètes : 57,140
Features : FeatureExtractor(10 dims: value, mask, interpolation, rolling_stats_w6_12, hourly_profile)
Nombre de courbes : 57,140

Analyse des trous réels :
  - Petits (

Masking:  66%|██████▌   | 37578/57140 [01:00<00:44, 438.92it/s]

## Prédictions

In [ ]:
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

def predict(model, X_df, scaler, config, batch_size=64):
    """
    Prédire les valeurs manquantes avec le FeatureExtractor de la config
    
    Args:
        model: modèle BiLSTM entraîné
        X_df: DataFrame avec colonnes à prédire
        scaler: StandardScaler utilisé pendant l'entraînement
        config: Config object (contient feature_extractor)
        batch_size: taille des batchs pour prédiction
    
    Returns:
        result: DataFrame avec valeurs imputées
    """
    from train.datasets.TimeSeries import TimeSeriesDataset
    
    print("\n" + "="*60)
    print("PRÉDICTION")
    print("="*60)
    print(f"Features utilisées : {config.feature_extractor}")
    print(f"Colonnes à prédire : {len(X_df.columns)}")

    # 1. Créer le dataset avec FeatureExtractor
    dataset = TimeSeriesDataset(
        X=X_df,
        y=None,
        feature_extractor=config.feature_extractor,  # ← Depuis config
        scaler=scaler,
        fit_scaler=False
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # 2. Prédiction
    model.eval()
    all_predictions = []

    with torch.no_grad():
        for batch_data in tqdm(loader, desc="Predicting"):
            # Gérer le cas où on a (x, mask) ou juste x
            if isinstance(batch_data, (list, tuple)):
                if len(batch_data) == 2:
                    x, mask = batch_data
                else:
                    x = batch_data[0]
                    mask = None
            else:
                x = batch_data
                mask = None
            
            x = x.to(config.device)
            if mask is not None:
                mask = mask.to(config.device)
            
            pred = model(x, mask)
            all_predictions.append(pred.cpu().numpy())

    # 3. Recomposition batchs → matrice complète
    predictions = np.concatenate(all_predictions, axis=0)  # (n_series, timesteps, 1)
    predictions = predictions.squeeze(-1).T                # → (timesteps, n_series)

    # 4. Dénormalisation
    predictions_denorm = scaler.inverse_transform(predictions.reshape(-1, 1))
    predictions_denorm = predictions_denorm.reshape(predictions.shape)

    pred_df = pd.DataFrame(
        predictions_denorm,
        index=X_df.index,
        columns=X_df.columns
    )

    # 5. Remplacement des NaNs uniquement
    result = X_df.copy()
    nan_mask = X_df.isna()
    result[nan_mask] = pred_df[nan_mask]
    
    n_imputed = nan_mask.sum().sum()
    print(f"\n✓ Prédiction terminée : {n_imputed:,} valeurs imputées")

    return result

In [ ]:
print(config.feature_extractor)

In [ ]:
# === 1. Extraire les colonnes à prédire ===
X_test_holed_cols = [c for c in X_test.columns if c.startswith("holed")]
X_test_holed = X_test[X_test_holed_cols]

print(f"Colonnes à prédire : {len(X_test_holed_cols)}")

# === 2. Prédire avec ton modèle entraîné ===

X_test_imputed = predict(
    model=model,           # Ton modèle entraîné
    X_df=X_test_holed,    # Colonnes à trous
    scaler=scaler,        # Scaler de l'entraînement
    config=config,        # Ta config avec feature_extractor
    batch_size=64
)

# === 3. Sauvegarder les prédictions ===
X_test_imputed.to_csv("predictions_bilstm_8features.csv")
print("✓ Prédictions sauvegardées dans predictions_bilstm.csv")

## Analyse des erreurs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

y_pred = predict(
    model=model,           # Ton modèle entraîné
    X_df=X_tr[holed_cols],    # Colonnes à trous
    scaler=scaler,        # Scaler de l'entraînement
    config=config,        # Ta config avec feature_extractor
    batch_size=64
)

# Après prédiction sur y_train
errors = np.abs(Y_tr - y_pred)  # MAE par point
